# ERS Weight Calibration & Visualisation

This notebook has four sections:

1. **Data Audit** — how much data do we have per component, per sector?
2. **Baseline Scores** — run equal weights, inspect the distribution
3. **Weight Calibration** — test weights against R1 historical enforcement actions
4. **Sensitivity Analysis** — see which components drive the score most

Run cells top-to-bottom. Requires `ers_scoring.py` in the same directory.

In [5]:
%pip install scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 6.0 MB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from datetime import date, timedelta
from scipy.optimize import minimize
from scipy.stats import spearmanr
from dotenv import load_dotenv

load_dotenv()

from ers_scoring import (
    engine, ERSWeights, compute_ers,
    run_sector_scores, df_from_query,
    score_regulatory, score_legislative, score_political,
    score_judicial, score_media, score_complaint_volume,
    DEFAULT_SECTORS, create_score_tables,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('✓ Imports OK')

✓ Imports OK


---
## Section 1 — Data Audit
Before scoring anything, understand how much data each component has.
Empty components will be excluded from the weighted average (weight redistributed to others).

In [7]:
# ── Row counts per table ──────────────────────────────────────────────────────
tables = {
    'R1 Enforcement Register':      'r1_enforcement_register',
    'R2 ICO News':                   'r2_ico_news',
    'R3 ICO Consultations':          'r3_ico_consultations',
    'L1 Bills':                      'l1_bills_in_parliament',
    'L2 Statutory Instruments':      'l2_statutory_instruments',
    'P1 Government Speeches':        'p1_government_speeches',
    'P6 Parliamentary Q&A':          'p6_parliamentary_qa',
    'J1 Supreme Court':              'j1_supreme_court',
    'J2 Court of Appeal':            'j2_court_of_appeal',
    'J3 Information Rights Tribunal':'j3_information_rights_tribunal',
    'J4 High Court':                 'j4_high_court',
    'M1 NGO Activity':               'm1_ngo_activity',
    'M2 Media Press':                'm2_media_press',
    'I1 Volume Statistics':          'i1_volume_statistics',
    'I2 Volume Scores':              'i2_volume_scores',
}

audit_rows = []
for label, tbl in tables.items():
    try:
        df = df_from_query(f'SELECT COUNT(*) AS n FROM {tbl}')
        n = int(df['n'].iloc[0])
    except Exception:
        n = 0
    audit_rows.append({'Table': label, 'Rows': n})

audit_df = pd.DataFrame(audit_rows)
display(audit_df.style.bar(subset=['Rows'], color='steelblue').format({'Rows': '{:,}'}))

,Table,Rows
0,R1 Enforcement Register,0
1,R2 ICO News,0
2,R3 ICO Consultations,0
3,L1 Bills,0
4,L2 Statutory Instruments,0
5,P1 Government Speeches,0
6,P6 Parliamentary Q&A,0
7,J1 Supreme Court,0
8,J2 Court of Appeal,0
9,J3 Information Rights Tribunal,0


In [8]:
# ── Date ranges per key table ─────────────────────────────────────────────────
date_tables = {
    'R1 Enforcement': ('r1_enforcement_register', 'action_date'),
    'L1 Bills':       ('l1_bills_in_parliament',  'event_date'),
    'P1 Speeches':    ('p1_government_speeches',  'speech_date'),
    'M1 NGO':         ('m1_ngo_activity',          'publication_date'),
    'M2 Press':       ('m2_media_press',            'publication_date'),
    'J3 Tribunal':    ('j3_information_rights_tribunal', 'decision_date'),
}

range_rows = []
for label, (tbl, col) in date_tables.items():
    try:
        df = df_from_query(f'SELECT MIN({col}) AS min_d, MAX({col}) AS max_d FROM {tbl}')
        range_rows.append({
            'Component': label,
            'Earliest': str(df['min_d'].iloc[0])[:10],
            'Latest':   str(df['max_d'].iloc[0])[:10],
        })
    except Exception:
        range_rows.append({'Component': label, 'Earliest': 'N/A', 'Latest': 'N/A'})

print('Date ranges per table:')
display(pd.DataFrame(range_rows))

Date ranges per table:


,Component,Earliest,Latest
0,R1 Enforcement,None,None
1,L1 Bills,None,None
2,P1 Speeches,None,None
3,M1 NGO,None,None
4,M2 Press,None,None
5,J3 Tribunal,None,None


---
## Section 2 — Baseline Scores (Equal Weights)
Run the scorer with equal weights across all sectors for the last 90 days.

In [ ]:
# ── Compute component scores for each sector ──────────────────────────────────
WINDOW_DAYS = 90
window_end   = date.today()
window_start = window_end - timedelta(days=WINDOW_DAYS)

print(f'Scoring window: {window_start} → {window_end}')
print(f'Sectors: {len(DEFAULT_SECTORS)}')

scorer_fns = {
    'regulatory':  score_regulatory,
    'legislative': score_legislative,
    'political':   score_political,
    'judicial':    score_judicial,
    'media':       score_media,
    'complaint':   score_complaint_volume,
}

component_data = []
for sector in DEFAULT_SECTORS:
    row = {'sector': sector}
    for comp, fn in scorer_fns.items():
        try:
            cs = fn('sector', sector, window_start, window_end)
            row[comp] = cs.normalised_score
            row[f'{comp}_count'] = cs.signal_count
            row[f'{comp}_hasdata'] = cs.has_data
        except Exception as e:
            row[comp] = 0.0
            row[f'{comp}_count'] = 0
            row[f'{comp}_hasdata'] = False
    component_data.append(row)

comp_df = pd.DataFrame(component_data)
comp_cols = list(scorer_fns.keys())
print('\nNormalised component scores (0–1):')
display(comp_df[['sector'] + comp_cols].round(3))

In [ ]:
# ── Compute ERS with equal weights ───────────────────────────────────────────
equal_weights = ERSWeights()   # defaults: each component ≈ equal

ers_rows = []
for sector in DEFAULT_SECTORS:
    composite = compute_ers('sector', sector, equal_weights, WINDOW_DAYS, window_end)
    ers_rows.append({
        'sector': sector,
        'ers_score': composite.ers_score,
        'ers_band': composite.ers_band,
        'data_completeness': composite.data_completeness,
    })

ers_df = pd.DataFrame(ers_rows).sort_values('ers_score', ascending=False).reset_index(drop=True)
print('ERS Sector Rankings (equal weights):')
display(ers_df.style.background_gradient(subset=['ers_score'], cmap='RdYlGn_r').format(
    {'ers_score': '{:.1f}', 'data_completeness': '{:.0%}'}
))

In [ ]:
# ── Heatmap: component scores by sector ──────────────────────────────────────
heat_df = comp_df.set_index('sector')[comp_cols]

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(
    heat_df, annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.4, ax=ax, vmin=0, vmax=1,
    cbar_kws={'label': 'Normalised score (0–1)'}
)
ax.set_title(f'Component Scores by Sector — {window_start} to {window_end}', fontsize=14, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('ers_component_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bar chart: ERS ranking ────────────────────────────────────────────────────
band_colours = {
    'Critical': '#d32f2f', 'High': '#f57c00',
    'Medium':   '#fbc02d', 'Low': '#388e3c', 'Minimal': '#1976d2'
}
colours = ers_df['ers_band'].map(band_colours)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(ers_df['sector'][::-1], ers_df['ers_score'][::-1], color=colours[::-1])
ax.set_xlabel('ERS Score (0–100)')
ax.set_title('Enforcement Risk Score — Sector Rankings (Equal Weights)', fontsize=13)
ax.axvline(75, color='#d32f2f', linestyle='--', alpha=0.5, label='Critical threshold (75)')
ax.axvline(55, color='#f57c00', linestyle='--', alpha=0.5, label='High threshold (55)')
ax.axvline(35, color='#fbc02d', linestyle='--', alpha=0.5, label='Medium threshold (35)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ers_sector_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Weight Calibration

**Strategy**: use R1 historical enforcement actions as ground truth.
For each enforcement action we know:
- Which sector/org was penalised
- The date of the action

We score each entity in a 90-day window *before* the enforcement action,
then optimise weights so that entities that were actually enforced against
score higher than those that weren't.

**Metric**: rank correlation (Spearman) between ERS and actual enforcement frequency.

> ⚠️ This requires enough R1 data. Check the row count in Section 1 first.
> If R1 is sparse, use the expert weight grid search below instead.

In [ ]:
# ── Load R1 enforcement history ───────────────────────────────────────────────
r1_df = df_from_query("""
    SELECT org_name, org_type, action_date, action_type,
           penalty_gbp, severity_tier
    FROM r1_enforcement_register
    ORDER BY action_date
""")

print(f'R1 enforcement actions loaded: {len(r1_df)}')
if len(r1_df) == 0:
    print('⚠️  No R1 data yet — run the regulatory scraper first.')
    print('    Skip to Section 3b (expert weights) for now.')
else:
    r1_df['action_date'] = pd.to_datetime(r1_df['action_date'])
    print(f'Date range: {r1_df["action_date"].min().date()} → {r1_df["action_date"].max().date()}')
    print(f'Unique orgs: {r1_df["org_name"].nunique()}')
    display(r1_df.head(10))

In [ ]:
# ── Build ground-truth sector enforcement frequency ───────────────────────────
# For each sector, count how many enforcement actions occurred.
# We use this as the ground truth to calibrate weights.

def sector_enforcement_count(sector: str, df: pd.DataFrame) -> int:
    """Count R1 rows that plausibly relate to a given sector."""
    kw = sector.split()[0].lower()
    mask = df['org_type'].str.lower().str.contains(kw, na=False)
    return int(mask.sum())

gt_rows = []
for sector in DEFAULT_SECTORS:
    count = sector_enforcement_count(sector, r1_df) if len(r1_df) > 0 else 0
    gt_rows.append({'sector': sector, 'enforcement_count': count})

gt_df = pd.DataFrame(gt_rows).sort_values('enforcement_count', ascending=False)
print('Ground truth — enforcement action counts by sector:')
display(gt_df)

In [ ]:
# ── Objective function: maximise Spearman correlation between ERS and enforcement count ──

def ers_from_weights_vector(w_vec, comp_df, gt_df):
    """
    Given a 6-element weight vector and pre-computed component scores,
    return the ERS for each sector.
    """
    w = w_vec / w_vec.sum()   # ensure they sum to 1
    comp_cols = ['regulatory','legislative','political','judicial','media','complaint']
    scores = (comp_df[comp_cols].values * w).sum(axis=1) * 100
    return scores


def objective(w_vec):
    """Minimise negative Spearman correlation (= maximise correlation)."""
    ers_scores = ers_from_weights_vector(w_vec, comp_df, gt_df)
    # Align on sector
    merged = comp_df[['sector']].copy()
    merged['ers'] = ers_scores
    merged = merged.merge(gt_df, on='sector')
    if merged['enforcement_count'].std() == 0:
        return 0.0   # no variance in ground truth — can't calibrate
    corr, _ = spearmanr(merged['ers'], merged['enforcement_count'])
    return -corr   # minimise negative = maximise correlation


# Only run optimisation if we have enough R1 data
if len(r1_df) > 10 and gt_df['enforcement_count'].std() > 0:
    print('Running weight optimisation...')
    # Start from equal weights, constrained to [0.05, 0.50] each, sum = 1
    w0 = np.array([1/6] * 6)
    bounds = [(0.05, 0.50)] * 6
    constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]

    result = minimize(
        objective, w0, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'maxiter': 500, 'ftol': 1e-9}
    )

    opt_w = result.x / result.x.sum()
    comp_labels = ['regulatory','legislative','political','judicial','media','complaint']

    opt_weights = ERSWeights(
        regulatory  = float(opt_w[0]),
        legislative = float(opt_w[1]),
        political   = float(opt_w[2]),
        judicial    = float(opt_w[3]),
        media       = float(opt_w[4]),
        complaint   = float(opt_w[5]),
        version     = 'v2_calibrated_spearman',
    )

    final_corr = -result.fun
    print(f'\nOptimised weights (Spearman corr = {final_corr:.3f}):')
    for label, w in zip(comp_labels, opt_w):
        print(f'  {label:15s}: {w:.4f}')
    print(f'\nSuccess: {result.success}  |  Message: {result.message}')
else:
    print('⚠️  Not enough R1 data variance for optimisation yet.')
    print('    Proceed to Section 3b (expert weight grid search).')
    opt_weights = None

### Section 3b — Expert Weight Grid Search
When R1 data is sparse, test a set of manually defined weight scenarios and compare the resulting rankings.

In [ ]:
# ── Define weight scenarios to compare ───────────────────────────────────────
weight_scenarios = {
    'Equal':             ERSWeights(0.167, 0.167, 0.167, 0.167, 0.167, 0.167, version='equal'),
    'Regulatory-heavy':  ERSWeights(0.35,  0.20,  0.15,  0.15,  0.08,  0.07,  version='reg_heavy'),
    'Legislative-heavy': ERSWeights(0.20,  0.35,  0.15,  0.15,  0.08,  0.07,  version='leg_heavy'),
    'Judicial-heavy':    ERSWeights(0.25,  0.15,  0.10,  0.30,  0.10,  0.10,  version='jud_heavy'),
    'Signal-balanced':   ERSWeights(0.25,  0.20,  0.15,  0.20,  0.10,  0.10,  version='sig_balanced'),
    'Volume-heavy':      ERSWeights(0.20,  0.15,  0.10,  0.20,  0.10,  0.25,  version='vol_heavy'),
}

# For each scenario, compute ERS for each sector
scenario_results = {}
comp_cols = ['regulatory','legislative','political','judicial','media','complaint']

for scenario_name, w in weight_scenarios.items():
    w_arr = np.array([w.regulatory, w.legislative, w.political,
                      w.judicial,   w.media,       w.complaint])
    w_arr = w_arr / w_arr.sum()
    scores = (comp_df[comp_cols].values * w_arr).sum(axis=1) * 100
    scenario_results[scenario_name] = scores

scenario_df = comp_df[['sector']].copy()
for name, scores in scenario_results.items():
    scenario_df[name] = scores.round(1)

print('ERS scores under different weight scenarios:')
display(scenario_df.set_index('sector').style.background_gradient(cmap='RdYlGn_r').format('{:.1f}'))

In [ ]:
# ── Spearman correlation between each scenario and enforcement frequency ──────
if len(r1_df) > 0:
    corr_rows = []
    for name in weight_scenarios:
        merged = scenario_df[['sector', name]].merge(gt_df, on='sector')
        if merged['enforcement_count'].std() > 0:
            corr, pval = spearmanr(merged[name], merged['enforcement_count'])
        else:
            corr, pval = None, None
        corr_rows.append({'Scenario': name, 'Spearman ρ': corr, 'p-value': pval})

    corr_df = pd.DataFrame(corr_rows).sort_values('Spearman ρ', ascending=False)
    print('Correlation of each weight scenario with R1 enforcement frequency:')
    display(corr_df.style.background_gradient(
        subset=['Spearman ρ'], cmap='RdYlGn').format({'Spearman ρ': '{:.3f}', 'p-value': '{:.3f}'}
    ))
else:
    print('No R1 data to correlate against yet — compare scenarios visually above.')

In [ ]:
# ── Rank stability plot: how much do sector rankings change across scenarios? ──
rank_df = scenario_df.set_index('sector').rank(ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
for sector in DEFAULT_SECTORS:
    ax.plot(list(weight_scenarios.keys()), rank_df.loc[sector], marker='o', label=sector, alpha=0.8)

ax.invert_yaxis()
ax.set_ylabel('Rank (1 = highest risk)')
ax.set_title('Sector Rank Stability Across Weight Scenarios', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('ers_rank_stability.png', dpi=150, bbox_inches='tight')
plt.show()

# Sectors with stable rank across all scenarios are the most reliable
rank_std = rank_df.std(axis=1).sort_values()
print('\nRank stability (low std = stable across all weight scenarios):')
display(rank_std.rename('Rank Std Dev').round(2).to_frame())

---
## Section 4 — Sensitivity Analysis

Which components drive the composite score the most?
Useful for: (a) prioritising data collection, (b) validating that the model makes intuitive sense.

In [ ]:
# ── One-at-a-time sensitivity: set each weight to 0, see score change ─────────
base_weights = ERSWeights()   # equal weights
base_arr = np.array([1/6] * 6)

comp_cols = ['regulatory','legislative','political','judicial','media','complaint']
base_scores = (comp_df[comp_cols].values * base_arr).sum(axis=1) * 100

sensitivity_rows = []
for i, comp in enumerate(comp_cols):
    # Zero out one weight, redistribute evenly to others
    w_zeroed = base_arr.copy()
    w_zeroed[i] = 0
    if w_zeroed.sum() > 0:
        w_zeroed = w_zeroed / w_zeroed.sum()
    zeroed_scores = (comp_df[comp_cols].values * w_zeroed).sum(axis=1) * 100
    avg_change = (zeroed_scores - base_scores).abs().mean()
    sensitivity_rows.append({'Component': comp, 'Avg Score Change When Zeroed': avg_change})

sens_df = pd.DataFrame(sensitivity_rows).sort_values('Avg Score Change When Zeroed', ascending=False)
print('Sensitivity: average change in ERS when each component weight is set to zero')
display(sens_df.style.bar(subset=['Avg Score Change When Zeroed'], color='coral').format(
    {'Avg Score Change When Zeroed': '{:.2f}'}
))

In [ ]:
# ── Feature importance: correlation of each component with composite ERS ──────
base_ers_series = pd.Series(base_scores, index=comp_df['sector'])

corr_data = []
for comp in comp_cols:
    comp_series = comp_df.set_index('sector')[comp]
    corr = comp_series.corr(base_ers_series)
    corr_data.append({'Component': comp, 'Pearson r with ERS': corr})

corr_feat_df = pd.DataFrame(corr_data).sort_values('Pearson r with ERS', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sensitivity
axes[0].barh(sens_df['Component'][::-1], sens_df['Avg Score Change When Zeroed'][::-1], color='coral')
axes[0].set_xlabel('Avg ERS Change (points)')
axes[0].set_title('Component Sensitivity\n(score change when weight → 0)')

# Correlation
colours_corr = ['#2196F3' if v >= 0 else '#F44336' for v in corr_feat_df['Pearson r with ERS']]
axes[1].barh(corr_feat_df['Component'][::-1], corr_feat_df['Pearson r with ERS'][::-1], color=colours_corr[::-1])
axes[1].axvline(0, color='gray', linewidth=0.8)
axes[1].set_xlabel('Pearson r')
axes[1].set_title('Component–ERS Correlation\n(with equal weights)')

plt.tight_layout()
plt.savefig('ers_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Score breakdown for the highest-risk sector ───────────────────────────────
top_sector = ers_df.iloc[0]['sector']
top_row = comp_df[comp_df['sector'] == top_sector][comp_cols].iloc[0]
equal_w = np.array([1/6] * 6)

fig, ax = plt.subplots(figsize=(8, 5))
colours_radar = ['#1976D2','#388E3C','#F57C00','#7B1FA2','#D32F2F','#0097A7']
contributions = top_row.values * equal_w * 100

bars = ax.bar(comp_cols, contributions, color=colours_radar)
ax.set_title(f'ERS Score Breakdown — {top_sector}\n(equal weights, {window_start} to {window_end})', fontsize=12)
ax.set_ylabel('Contribution to ERS (points)')
ax.set_ylim(0, 30)

for bar, val in zip(bars, contributions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('ers_breakdown_top_sector.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total ERS: {contributions.sum():.1f} / 100')

---
## Section 5 — Export Final Weights & Save to DB

Once you are happy with a weight set, run this cell to:
1. Print the weights to copy into `ers_scoring.py`
2. Compute and save scores for all sectors to the `ers_composite_scores` table

In [ ]:
# ── Choose your final weights here ───────────────────────────────────────────
# Options:
#   a) opt_weights  — calibrated by Spearman optimisation (if R1 data available)
#   b) One of the weight_scenarios dict entries above
#   c) Manual weights — set your own below

FINAL_WEIGHTS = ERSWeights(
    regulatory  = 0.25,
    legislative = 0.20,
    political   = 0.15,
    judicial    = 0.20,
    media       = 0.10,
    complaint   = 0.10,
    version     = 'v1_signal_balanced',
)

FINAL_WEIGHTS.validate()
print('Final weights:')
for k, v in FINAL_WEIGHTS.as_dict().items():
    print(f'  {k:15s}: {v:.4f}')
print(f'\nVersion tag: {FINAL_WEIGHTS.version}')
print(f'Sum: {sum(FINAL_WEIGHTS.as_dict().values()):.4f}')

In [ ]:
# ── Save scores to DB ─────────────────────────────────────────────────────────
create_score_tables()
run_sector_scores(FINAL_WEIGHTS, window_days=WINDOW_DAYS)
print('\n✓ Scores saved to ers_composite_scores table')

# Verify
saved = df_from_query("""
    SELECT entity_id, ers_score, ers_band, data_completeness, computed_at
    FROM ers_composite_scores
    WHERE entity_type = 'sector'
    ORDER BY ers_score DESC
""")
display(saved)

In [ ]:
# ── Print ERSWeights snippet to paste into ers_scoring.py ────────────────────
print('Copy this into the ERSWeights dataclass defaults in ers_scoring.py:')
print('─' * 50)
print(f'@dataclass')
print(f'class ERSWeights:')
for k, v in FINAL_WEIGHTS.as_dict().items():
    print(f'    {k:12s}: float = {v:.4f}')
print(f'    version    : str   = "{FINAL_WEIGHTS.version}"')